# Analisi Dati Cinematici — VRula BodyTracking

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.stats import shapiro, ttest_rel, wilcoxon, mannwhitneyu, probplot
from scipy.cluster.vq import kmeans2
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D

script_dir  = os.path.join(os.path.expanduser("~"), "Desktop", "VRula", "BodyTracking", "separated_scripts")
figures_dir = os.path.join(script_dir, "output_analysis", "figures")
os.makedirs(figures_dir, exist_ok=True)
dataset_path = os.path.join(script_dir, "output_analysis", "analysis_dataset.csv")
print("script_dir:", script_dir)

## 0. Caricamento dati

In [ ]:
df = pd.read_csv(dataset_path)
df["Subject_ID"]  = df["Subject_ID"].astype("category")
df["Condition"]   = pd.Categorical(df["Condition"], categories=["baseline", "workload"], ordered=True)
df["Order_Group"] = df["Order_Group"].astype("category")
df["Dom_Hand"]    = df["Dom_Hand"].astype("category")
print(f"N soggetti: {df['Subject_ID'].nunique()}")
print(f"N righe:    {len(df)}")
df.head()

## 1. Variabili cinematiche e costanti

In [ ]:
kinematics_vars = [
    "SPARC_Dom", "Jerk_Dom", "VelInv_Dom", "Dwell_Dom",
    "Vel_Mean_Dom", "Vel_Peak_Dom", "Vel_SD_Dom", "PathLen_Dom",
    "RULA_Mean",
    "Wr_Flex_Mean_Dom", "Wr_Flex_SD_Dom",
    "Wr_Dev_Mean_Dom",  "Wr_Dev_SD_Dom",
    "UA_Mean_Dom",      "SD_UA_Dom",
    "LA_Mean_Dom",      "SD_LA_Dom",
    "Neck_Sag_Mean",    "Neck_Sag_SD",
    "Trunk_Sag_Mean",   "Trunk_Sag_SD",
]

COLORS       = {"baseline": "#4C9BE8", "workload": "#E8754C"}
ORDER_COLORS = {"A->B": "#5B8DB8", "B->A": "#C97B3A"}
COND_COLORS  = {"baseline": "#4C9BE8", "workload": "#E8754C"}
CONDS        = ["baseline", "workload"]
VARS         = ["SPARC_Dom", "Jerk_Dom"]
subjects     = df["Subject_ID"].cat.categories
subj_colors  = plt.cm.tab20(np.linspace(0, 1, len(subjects)))
rng          = np.random.default_rng(42)
print(f"Variabili cinematiche: {len(kinematics_vars)}")

## 2. Statistiche descrittive

In [ ]:
def descrittive(data, vars_list):
    rows = []
    for var in vars_list:
        x = pd.to_numeric(data[var], errors="coerce")
        rows.append({"variabile": var, "n": x.notna().sum(), "mean": x.mean(),
                     "sd": x.std(ddof=1), "min": x.min(), "q1": x.quantile(0.25),
                     "median": x.median(), "q3": x.quantile(0.75), "max": x.max()})
    return pd.DataFrame(rows).set_index("variabile").round(4)

print("--- BASELINE ---")
display(descrittive(df[df["Condition"] == "baseline"], kinematics_vars))
print("--- WORKLOAD ---")
display(descrittive(df[df["Condition"] == "workload"], kinematics_vars))

## 3. Rilevamento outlier (+/-2 SD per condizione)

In [ ]:
soglie_rows  = []
outlier_rows = []

for condition in ["baseline", "workload"]:
    idx_cond = df[df["Condition"] == condition].index
    for var in kinematics_vars:
        x     = pd.to_numeric(df.loc[idx_cond, var], errors="coerce")
        mu, sigma = x.mean(), x.std(ddof=1)
        lower, upper = mu - 2*sigma, mu + 2*sigma
        soglie_rows.append({"condition": condition, "variabile": var,
                            "mean": round(mu,4), "sd": round(sigma,4),
                            "lower_2sd": round(lower,4), "upper_2sd": round(upper,4)})
        mask = x.notna() & ((x < lower) | (x > upper))
        for idx in df.loc[idx_cond][mask].index:
            outlier_rows.append({"variabile": var, "Subject_ID": df.at[idx,"Subject_ID"],
                                 "Condition": condition, "valore": round(x[idx],4),
                                 "lower_2sd": round(lower,4), "upper_2sd": round(upper,4)})

soglie_df   = pd.DataFrame(soglie_rows).set_index(["condition","variabile"])
soglie_dict = {(r["condition"], r["variabile"]): (r["lower_2sd"], r["upper_2sd"]) for r in soglie_rows}
display(soglie_df)

if outlier_rows:
    outlier_df = pd.DataFrame(outlier_rows)
    print(f"Totale outlier: {len(outlier_df)}")
    display(outlier_df)
else:
    print("Nessun outlier trovato.")
    outlier_df = pd.DataFrame()

## 4. Sostituzione outlier con la media (per condizione)

In [ ]:
df_clean = df.copy()

for condition in ["baseline", "workload"]:
    idx_cond = df_clean[df_clean["Condition"] == condition].index
    for var in kinematics_vars:
        x = pd.to_numeric(df_clean.loc[idx_cond, var], errors="coerce")
        mu, sigma = x.mean(), x.std(ddof=1)
        is_out = x.notna() & ((x < mu - 2*sigma) | (x > mu + 2*sigma))
        if is_out.sum() > 0:
            mean_clean = x[~is_out].mean()
            df_clean[var] = df_clean[var].astype(float)
            df_clean.loc[idx_cond[is_out], var] = mean_clean
            print(f"  [{condition}] {var}: {is_out.sum()} outlier -> mean={mean_clean:.4f}")

# order_map disponibile per le sezioni successive
order_map = (df_clean[df_clean["Condition"] == "baseline"]
             .set_index("Subject_ID")["Order_Group"].astype(str))

print("\nDescrittive dopo pulizia:")
for condition in ["baseline", "workload"]:
    print(f"\n--- {condition.upper()} ---")
    display(descrittive(df_clean[df_clean["Condition"] == condition], kinematics_vars))

## 5. Visualizzazioni

### 5a. Boxplot tutte le variabili (dati originali + linee ±2 SD)

In [ ]:
ncols = 4
nrows = -(-len(kinematics_vars) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.5))
axes = axes.flatten()

for i, var in enumerate(kinematics_vars):
    ax = axes[i]
    data_orig = [df.loc[df["Condition"]==c, var].dropna().values.astype(float) for c in CONDS]
    bp = ax.boxplot(data_orig, patch_artist=True, widths=0.5,
                    medianprops=dict(color="black", linewidth=2), sym="")
    for patch, color in zip(bp["boxes"], COLORS.values()):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    for j, (vals, color) in enumerate(zip(data_orig, COLORS.values()), start=1):
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(j+jitter, vals, color=color, s=25, zorder=3, edgecolors="white", linewidths=0.4)
    for j, cond in enumerate(CONDS, start=1):
        lower, upper = soglie_dict[(cond, var)]
        ax.hlines([lower, upper], j-0.3, j+0.3, colors="red", linestyles="dashed", linewidth=1.2)
    ax.set_xticks([1,2]); ax.set_xticklabels(["Baseline","Workload"], fontsize=9)
    ax.set_title(var, fontsize=9, fontweight="bold")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

for ax in axes[len(kinematics_vars):]: ax.set_visible(False)
fig.suptitle("Boxplot variabili cinematiche — dati originali (linee rosse = +/-2 SD)",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "boxplots_cinematiche.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: boxplots_cinematiche.png")

### 5b. SPARC e Jerk — boxplot, paired lines, violin (dati puliti)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, var in enumerate(VARS):
    b_cln  = df_clean.loc[df_clean["Condition"]=="baseline",  var].values.astype(float)
    wl_cln = df_clean.loc[df_clean["Condition"]=="workload",  var].values.astype(float)

    ax = axes[row, 0]
    bp = ax.boxplot([b_cln, wl_cln], patch_artist=True, widths=0.5,
                    medianprops=dict(color="black", linewidth=2), sym="")
    for patch, color in zip(bp["boxes"], COLORS.values()):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    for j, (vals, color) in enumerate(zip([b_cln, wl_cln], COLORS.values()), start=1):
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(j+jitter, vals, color=color, s=40, zorder=3, edgecolors="white", linewidths=0.5)
    for j, cond in enumerate(CONDS, start=1):
        lower, upper = soglie_dict[(cond, var)]
        ax.hlines([lower, upper], j-0.3, j+0.3, colors="red", linestyles="dashed", linewidth=1.5)
    ax.set_xticks([1,2]); ax.set_xticklabels(["Baseline","Workload"])
    ax.set_title("Boxplot (dati puliti)"); ax.set_ylabel(var)

    ax = axes[row, 1]
    for subj, color in zip(subjects, subj_colors):
        v_b  = df_clean[(df_clean["Subject_ID"]==subj)&(df_clean["Condition"]=="baseline")][var].values
        v_wl = df_clean[(df_clean["Subject_ID"]==subj)&(df_clean["Condition"]=="workload")][var].values
        if len(v_b) and len(v_wl):
            ax.plot([0,1],[v_b[0],v_wl[0]], "-o", color=color, alpha=0.65, linewidth=1.4, markersize=6)
    ax.plot([0,1],[np.median(b_cln),np.median(wl_cln)], "k-o", linewidth=2.5, markersize=9, zorder=5, label="Mediana")
    ax.set_xticks([0,1]); ax.set_xticklabels(["Baseline","Workload"])
    ax.set_title("Paired lines (dati puliti)"); ax.legend(fontsize=8)

    ax = axes[row, 2]
    parts = ax.violinplot([b_cln, wl_cln], positions=[1,2], showmedians=True, showextrema=True)
    for pc, color in zip(parts["bodies"], COLORS.values()):
        pc.set_facecolor(color); pc.set_alpha(0.6)
    for part in ("cmedians","cmins","cmaxes","cbars"):
        parts[part].set_color("black"); parts[part].set_linewidth(1.2)
    for j, (vals, color) in enumerate(zip([b_cln, wl_cln], COLORS.values()), start=1):
        jitter = rng.uniform(-0.06, 0.06, size=len(vals))
        ax.scatter(j+jitter, vals, color=color, s=35, zorder=3, edgecolors="white", linewidths=0.5)
    ax.set_xticks([1,2]); ax.set_xticklabels(["Baseline","Workload"])
    ax.set_title("Violin plot (dati puliti)")

    for col in range(3):
        axes[row,col].spines["top"].set_visible(False)
        axes[row,col].spines["right"].set_visible(False)

fig.suptitle("SPARC_Dom e Jerk_Dom — Baseline vs Workload", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "sparc_jerk_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: sparc_jerk_overview.png")

## 6. Salvataggio dataset pulito

In [ ]:
out_path = os.path.join(script_dir, "output_analysis", "analysis_dataset_clean.csv")
df_clean.to_csv(out_path, index=False)
print(f"Salvato: {out_path}  |  Shape: {df_clean.shape}")

## 7. Normalità dei delta (Shapiro-Wilk)

### 7a. Shapiro-Wilk sui delta (workload - baseline)

In [ ]:
df_b  = df_clean[df_clean["Condition"]=="baseline"].set_index("Subject_ID")
df_wl = df_clean[df_clean["Condition"]=="workload"].set_index("Subject_ID")
common_subjects = df_b.index.intersection(df_wl.index)
delta = df_wl.loc[common_subjects, kinematics_vars] - df_b.loc[common_subjects, kinematics_vars]

normality_rows = []
for var in kinematics_vars:
    d = delta[var].dropna().values
    W, p = shapiro(d)
    normality_rows.append({"variabile": var, "W": round(W,4), "p": round(p,4),
                           "normale": "SI" if p > 0.05 else "NO"})

normality_df = pd.DataFrame(normality_rows)
display(normality_df.set_index("variabile"))

### 7b. QQ plot dei delta

In [ ]:
ncols = 4
nrows = -(-len(kinematics_vars) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*3.2))
axes = axes.flatten()

for i, var in enumerate(kinematics_vars):
    ax = axes[i]
    d  = delta[var].dropna().values
    (osm, osr), (slope, intercept, _) = probplot(d, dist="norm")
    ax.scatter(osm, osr, color="#4C9BE8", s=35, zorder=3)
    x_line = np.array([osm.min(), osm.max()])
    ax.plot(x_line, slope*x_line+intercept, "r--", linewidth=1.2)
    row = normality_df[normality_df["variabile"]==var].iloc[0]
    ax.set_title(f"{var}\np={row['p']:.3f}  {row['normale']}", fontsize=8, fontweight="bold")
    ax.set_xlabel("Quantili teorici", fontsize=7)
    ax.set_ylabel("Quantili campione", fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

for ax in axes[len(kinematics_vars):]: ax.set_visible(False)
fig.suptitle("QQ plot dei delta (workload - baseline)", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "qqplot_delta.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: qqplot_delta.png")

### 7c. Shapiro-Wilk per condizione (descrittivo)

In [ ]:
cond_normality_rows = []
for var in kinematics_vars:
    results = {}
    for cond, data in [("baseline", df_b), ("workload", df_wl)]:
        d = pd.to_numeric(data[var], errors="coerce").dropna().values
        W, p = shapiro(d)
        results[cond] = (round(W,4), round(p,4), "SI" if p>0.05 else "NO")
    W_b, p_b, n_b   = results["baseline"]
    W_wl, p_wl, n_wl = results["workload"]
    cond_normality_rows.append({"variabile": var,
                                "W_baseline": W_b,  "p_baseline":  p_b,  "normale_baseline":  n_b,
                                "W_workload":  W_wl, "p_workload":  p_wl, "normale_workload":  n_wl})

cond_normality_df = pd.DataFrame(cond_normality_rows)
display(cond_normality_df.set_index("variabile"))

## 8. Effetto condizione (paired t-test / Wilcoxon + FDR)

In [ ]:
condition_rows = []
N_subj = len(common_subjects)

for var in kinematics_vars:
    d       = delta[var].dropna().values
    normale = normality_df.loc[normality_df["variabile"]==var, "normale"].values[0]
    if normale == "SI":
        stat, p = ttest_rel(df_wl.loc[common_subjects, var].values.astype(float),
                            df_b.loc[common_subjects,  var].values.astype(float))
        test_name, effect, effect_name = "t-test", np.mean(d)/np.std(d, ddof=1), "d_z"
    else:
        stat, p = wilcoxon(df_wl.loc[common_subjects, var].values.astype(float),
                           df_b.loc[common_subjects,  var].values.astype(float),
                           alternative="two-sided")
        test_name, effect, effect_name = "Wilcoxon", 1-(2*stat)/(N_subj*(N_subj+1)/2), "r_rb"
    condition_rows.append({"variabile": var, "test": test_name, "statistica": round(stat,4),
                           "p": round(p,4), effect_name: round(effect,4),
                           "significativo": "SI" if p < 0.05 else "NO"})

condition_df = pd.DataFrame(condition_rows)
if "d_z"  not in condition_df: condition_df["d_z"]  = np.nan
if "r_rb" not in condition_df: condition_df["r_rb"] = np.nan
condition_df["effect"]  = condition_df["d_z"].fillna(condition_df["r_rb"])

reject, p_fdr, _, _ = multipletests(condition_df["p"], method="fdr_bh", alpha=0.05)
condition_df["p_fdr"]   = p_fdr.round(4)
condition_df["sig_fdr"] = reject
condition_df["sig"]     = condition_df["p"] < 0.05
sig_vars = set(condition_df.loc[condition_df["sig_fdr"], "variabile"])

print(f"Significative prima FDR: {condition_df['sig'].sum()}/21")
print(f"Significative dopo  FDR: {condition_df['sig_fdr'].sum()}/21")
display(condition_df[["variabile","test","p","sig","p_fdr","sig_fdr","effect"]].set_index("variabile"))

## 8b. Power Analysis post-hoc

Per le variabili **non significative** dopo FDR, la power analysis risponde alla domanda: *l'effetto non c'è, oppure il campione era troppo piccolo per rilevarlo?*

- Potenza ≥ 80% + n.s. → l'effetto è probabilmente assente (o molto piccolo)
- Potenza < 80% + n.s. → N insufficiente, effetto non rilevabile con questo campione

La potenza è calcolata con il paired t-test (TTestPower, `statsmodels`) usando il Cohen's d_z osservato e α=0.05, N=18.

In [ ]:
from statsmodels.stats.power import TTestPower

power_rows = []
ttest_power = TTestPower()

for _, row in condition_df.iterrows():
    var = row["variabile"]
    d   = delta[var].dropna().values
    d_z = np.mean(d) / np.std(d, ddof=1)
    n   = len(d)
    pwr = ttest_power.solve_power(effect_size=abs(d_z), nobs=n, alpha=0.05, alternative="two-sided")

    if row["sig_fdr"]:
        interp = "sig. FDR"
    elif pwr >= 0.80:
        interp = "potenza adeguata (effetto assente?)"
    elif pwr >= 0.50:
        interp = "potenza moderata (N insufficiente)"
    else:
        interp = "sotto-potenziata (N troppo piccolo)"

    power_rows.append({"variabile": var, "d_z": round(d_z,4), "N": n,
                       "power": round(pwr,4), "sig_fdr": row["sig_fdr"],
                       "interpretazione": interp})

power_df = pd.DataFrame(power_rows)

fig, ax = plt.subplots(figsize=(10, 7))
bar_cols = ["#2ECC71" if s else "#E74C3C" for s in power_df["sig_fdr"]]
ax.barh(power_df["variabile"], power_df["power"], color=bar_cols, alpha=0.85, edgecolor="white")
ax.axvline(0.80, color="black", linestyle="--", linewidth=1.5, label="80% (soglia convenzionale)")
ax.axvline(0.50, color="gray",  linestyle=":",  linewidth=1.2, label="50%")
ax.set_xlabel("Potenza osservata (alpha=0.05, N=18)", fontsize=10)
ax.set_title("Power Analysis post-hoc\nverde = sig FDR   rosso = non significativa", fontsize=10, fontweight="bold")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "power_analysis.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: power_analysis.png")
display(power_df.set_index("variabile"))

## 9. Grafici effetto condizione

In [ ]:
condition_df_sorted = condition_df.sort_values("effect")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

bar_colors = ["#E8754C" if s else "#AAAAAA" for s in condition_df_sorted["sig"]]
ax1.barh(condition_df_sorted["variabile"], condition_df_sorted["effect"],
         color=bar_colors, edgecolor="white", height=0.7)
ax1.axvline(0, color="black", linewidth=1)
for val in [0.2, 0.5, 0.8]:
    for sign in [1, -1]:
        ax1.axvline(sign*val, color="gray", linewidth=0.8, linestyle=":", alpha=0.6)
for _, row in condition_df_sorted.iterrows():
    if row["sig"]:
        x_pos = row["effect"] + (0.03 if row["effect"] >= 0 else -0.03)
        ax1.text(x_pos, row["variabile"], f"p={row['p']:.3f} ({row['test']})",
                 va="center", ha="left" if row["effect"] >= 0 else "right", fontsize=7)
ax1.set_xlabel("Effect size  (d_z | r_rb)", fontsize=9)
ax1.set_title("Effect size per variabile\n(rosso = p < 0.05)", fontsize=10, fontweight="bold")
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

neg_log_p = -np.log10(condition_df["p"])
def point_color(row):
    if row["sig_fdr"]: return "#C0392B"
    if row["sig"]:     return "#E8A44C"
    return "#AAAAAA"
point_colors = [point_color(row) for _, row in condition_df.iterrows()]
ax2.scatter(condition_df["effect"], neg_log_p, c=point_colors, s=80,
            edgecolors="white", linewidths=0.5, zorder=3)
ax2.axhline(-np.log10(0.05), color="gray", linestyle="dashed", linewidth=1.2, label="p=0.05 grezzo")
if condition_df["sig_fdr"].any():
    fdr_thr = condition_df.loc[condition_df["sig_fdr"], "p"].max()
    ax2.axhline(-np.log10(fdr_thr), color="red", linestyle="dashed", linewidth=1.2,
                label=f"soglia FDR (p={fdr_thr:.3f})")
ax2.axvline(0, color="black", linewidth=0.8)
for _, row in condition_df.iterrows():
    if row["sig"]:
        ax2.annotate(row["variabile"], xy=(row["effect"], -np.log10(row["p"])),
                     xytext=(5,3), textcoords="offset points", fontsize=7)
ax2.set_xlabel("Effect size", fontsize=9); ax2.set_ylabel("-log10(p)", fontsize=9)
ax2.set_title("Volcano plot\nrosso=FDR sig  arancione=p<0.05  grigio=n.s.", fontsize=10, fontweight="bold")
ax2.legend(fontsize=8); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("Effetto Condition (baseline vs workload) — N=18", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "condition_effect.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: condition_effect.png")

## 10. Order effect check

In [ ]:
delta_ord = delta.copy()
delta_ord["Order_Group"] = order_map.loc[delta_ord.index].values
grp_ab = delta_ord[delta_ord["Order_Group"] == "A->B"]
grp_ba = delta_ord[delta_ord["Order_Group"] == "B->A"]
print(f"A->B: n={len(grp_ab)}   B->A: n={len(grp_ba)}")

order_rows = []
for var in kinematics_vars:
    a = grp_ab[var].dropna().values.astype(float)
    b = grp_ba[var].dropna().values.astype(float)
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    order_rows.append({"variabile": var, "median_AB": round(np.median(a),4),
                       "median_BA": round(np.median(b),4), "U": U, "p": round(p,4),
                       "significativo": "SI" if p < 0.05 else "NO"})

order_df = pd.DataFrame(order_rows)
print(f"Variabili con order effect (p<0.05): {(order_df['p']<0.05).sum()}/{len(kinematics_vars)}")
display(order_df.set_index("variabile"))

In [ ]:
ncols = 4
nrows = -(-len(kinematics_vars) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.8, nrows*3.2))
axes = axes.flatten()

for i, var in enumerate(kinematics_vars):
    ax = axes[i]
    for x_pos, (grp_label, grp_data) in enumerate([("A->B", grp_ab), ("B->A", grp_ba)]):
        vals   = grp_data[var].dropna().values.astype(float)
        color  = ORDER_COLORS[grp_label]
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(x_pos+jitter, vals, color=color, s=45, zorder=3,
                   edgecolors="white", linewidths=0.4, label=grp_label)
        ax.plot([x_pos-0.15, x_pos+0.15], [np.median(vals)]*2, color=color, linewidth=2.5, zorder=4)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
    p_val = order_df.loc[order_df["variabile"]==var, "p"].values[0]
    ax.set_title(f"{var}\np={p_val:.3f}" + (" *" if p_val<0.05 else ""), fontsize=8, fontweight="bold")
    ax.set_xticks([0,1]); ax.set_xticklabels(["A->B","B->A"], fontsize=9)
    ax.set_ylabel("Delta (WL-Base)", fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    if i == 0: ax.legend(fontsize=7)

for ax in axes[len(kinematics_vars):]: ax.set_visible(False)
fig.suptitle("Order effect — delta per gruppo d'ordine\nLinea = mediana del gruppo",
             fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "order_effect.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: order_effect.png")

## 11. PCA sui delta

### 11a. Calcolo SVD

In [ ]:
delta_mat = delta[kinematics_vars].dropna()
subj_ids  = delta_mat.index.tolist()
X         = delta_mat.values.astype(float)
X_std     = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
scores   = U * S
loadings = Vt.T
var_exp  = (S**2) / np.sum(S**2)
cum_var  = np.cumsum(var_exp)
n_comp   = len(var_exp)

print(f"{'PC':<5} {'Var. spiegata':>15} {'Cumulata':>10}")
print("-"*33)
for i in range(min(6, n_comp)):
    print(f"PC{i+1:<3} {var_exp[i]*100:>14.1f}%  {cum_var[i]*100:>9.1f}%")

### 11b. Scree plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
pcs = np.arange(1, n_comp+1)
ax1.bar(pcs, var_exp*100, color="#4C9BE8", alpha=0.8, edgecolor="white")
ax1.set_xlabel("Componente"); ax1.set_ylabel("Varianza spiegata (%)")
ax1.set_title("Scree plot"); ax1.set_xticks(pcs)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

ax2.plot(pcs, cum_var*100, "o-", color="#E8754C", linewidth=2, markersize=6)
ax2.axhline(80, color="gray", linestyle="--", linewidth=1, label="80%")
ax2.set_xlabel("N componenti"); ax2.set_ylabel("Varianza cumulata (%)")
ax2.set_title("Varianza cumulata"); ax2.set_xticks(pcs); ax2.legend(fontsize=9)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("PCA sui delta — varianza spiegata", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "pca_scree.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: pca_scree.png")

### 11c. Biplot PC1 × PC2

In [ ]:
order_colors_subj = [ORDER_COLORS.get(
    str(order_map.loc[s]) if s in order_map.index else "A->B", "#888888"
) for s in subj_ids]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(scores[:,0], scores[:,1], c=order_colors_subj, s=80, zorder=3,
           edgecolors="white", linewidths=0.5)
for i, s in enumerate(subj_ids):
    ax.annotate(str(s), (scores[i,0], scores[i,1]), xytext=(4,4),
                textcoords="offset points", fontsize=8)

scale = np.max(np.abs(scores[:,:2])) * 0.9
for j, var in enumerate(kinematics_vars):
    lx, ly = loadings[j,0]*scale, loadings[j,1]*scale
    color  = "#C0392B" if var in sig_vars else "#888888"
    ax.annotate("", xy=(lx,ly), xytext=(0,0),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
    ax.text(lx*1.08, ly*1.08, var, fontsize=7, color=color, ha="center")

ax.axhline(0, color="gray", linewidth=0.5); ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% varianza)", fontsize=10)
ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% varianza)", fontsize=10)
ax.set_title("Biplot PCA\nFrecce rosse = sig FDR  |  colore = ordine", fontsize=10, fontweight="bold")
legend_elements = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor=ORDER_COLORS["A->B"], markersize=9, label="A->B"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=ORDER_COLORS["B->A"], markersize=9, label="B->A"),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "pca_biplot.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: pca_biplot.png")

### 11d. Heatmap loadings PC1–PC4

In [ ]:
n_show  = min(4, n_comp)
load_df = pd.DataFrame(loadings[:,:n_show], index=kinematics_vars,
                       columns=[f"PC{i+1}" for i in range(n_show)])

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(load_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Loading")
ax.set_xticks(range(n_show))
ax.set_xticklabels([f"PC{i+1}\n({var_exp[i]*100:.1f}%)" for i in range(n_show)], fontsize=9)
ax.set_yticks(range(len(kinematics_vars)))
ax.set_yticklabels([f"{'*' if v in sig_vars else ' '} {v}" for v in kinematics_vars], fontsize=8)
for i in range(len(kinematics_vars)):
    for j in range(n_show):
        val = load_df.values[i,j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(val)>0.5 else "black")
ax.set_title("Loadings PCA (prime 4 componenti)\n* = significativa dopo FDR",
             fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "pca_loadings.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: pca_loadings.png")

## 12. K-Means k=2 sulle condizioni

Verifica se le due condizioni (baseline/workload) sono separabili in modo non supervisionato nello spazio delle 21 variabili (36 righe totali).

In [ ]:
X_km      = df_clean[kinematics_vars].astype(float).values
y_true    = (df_clean["Condition"] == "workload").astype(int).values
subj_full = df_clean["Subject_ID"].values
cond_full = df_clean["Condition"].values
X_km_std  = (X_km - X_km.mean(axis=0)) / X_km.std(axis=0, ddof=1)

best_labels, best_inertia, best_centroids = None, np.inf, None
for seed in range(100):
    np.random.seed(seed)
    centroids, labels = kmeans2(X_km_std, 2, iter=300, minit="points")
    inertia = sum(np.sum((X_km_std[labels==k]-centroids[k])**2) for k in range(2))
    if inertia < best_inertia:
        best_inertia, best_labels, best_centroids = inertia, labels.copy(), centroids.copy()

if np.mean((1-best_labels)==y_true) > np.mean(best_labels==y_true):
    best_labels = 1 - best_labels

tp = int(np.sum((best_labels==1)&(y_true==1))); tn = int(np.sum((best_labels==0)&(y_true==0)))
fp = int(np.sum((best_labels==1)&(y_true==0))); fn = int(np.sum((best_labels==0)&(y_true==1)))
accuracy = (tp+tn)/len(y_true)

print(f"Accuracy: {accuracy*100:.1f}%")
print(f"\nMatrice di confusione:")
print(f"                   Pred baseline  Pred workload")
print(f"  True baseline         {tn:2d}              {fp:2d}")
print(f"  True workload         {fn:2d}              {tp:2d}")

errors = np.where(best_labels != y_true)[0]
if len(errors):
    print(f"\nRighe errate ({len(errors)}):")
    for idx in errors:
        print(f"  Soggetto {subj_full[idx]}, condizione {cond_full[idx]}")

In [ ]:
U_f, S_f, Vt_f = np.linalg.svd(X_km_std, full_matrices=False)
scores_f  = U_f * S_f
var_exp_f = (S_f**2) / np.sum(S_f**2)
correct_mask = best_labels == y_true

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6))

for cond, col in COND_COLORS.items():
    mask = cond_full == cond
    ax_l.scatter(scores_f[mask,0], scores_f[mask,1], c=col, s=70,
                 edgecolors="white", linewidths=0.5, label=cond, zorder=3)
for i in range(len(subj_full)):
    ax_l.annotate(str(subj_full[i]), (scores_f[i,0], scores_f[i,1]),
                  xytext=(3,3), textcoords="offset points", fontsize=7)
ax_l.axhline(0,color="gray",linewidth=0.4); ax_l.axvline(0,color="gray",linewidth=0.4)
ax_l.set_xlabel(f"PC1 ({var_exp_f[0]*100:.1f}%)", fontsize=10)
ax_l.set_ylabel(f"PC2 ({var_exp_f[1]*100:.1f}%)", fontsize=10)
ax_l.set_title("Condizione vera", fontsize=10, fontweight="bold")
ax_l.legend(fontsize=9); ax_l.spines["top"].set_visible(False); ax_l.spines["right"].set_visible(False)

markers = {"baseline": "o", "workload": "s"}
for cond, marker in markers.items():
    mask = cond_full == cond
    idx_c = np.where(mask & correct_mask)[0]
    idx_w = np.where(mask & ~correct_mask)[0]
    if len(idx_c): ax_r.scatter(scores_f[idx_c,0], scores_f[idx_c,1], c="#2ECC71", marker=marker,
                                 s=80, edgecolors="white", linewidths=0.5, zorder=3, label=f"{cond} corretto")
    if len(idx_w): ax_r.scatter(scores_f[idx_w,0], scores_f[idx_w,1], c="#E74C3C", marker=marker,
                                 s=100, edgecolors="black", linewidths=0.8, zorder=4, label=f"{cond} errato")
for i in range(len(subj_full)):
    ax_r.annotate(str(subj_full[i]), (scores_f[i,0], scores_f[i,1]),
                  xytext=(3,3), textcoords="offset points", fontsize=7)
ax_r.axhline(0,color="gray",linewidth=0.4); ax_r.axvline(0,color="gray",linewidth=0.4)
ax_r.set_xlabel(f"PC1 ({var_exp_f[0]*100:.1f}%)", fontsize=10)
ax_r.set_ylabel(f"PC2 ({var_exp_f[1]*100:.1f}%)", fontsize=10)
ax_r.set_title(f"K-Means (accuracy {accuracy*100:.1f}%)\ncerchio=baseline  quadrato=workload",
               fontsize=10, fontweight="bold")
ax_r.legend(fontsize=8, loc="best")
ax_r.spines["top"].set_visible(False); ax_r.spines["right"].set_visible(False)

fig.suptitle("K-Means k=2 — separazione baseline vs workload", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "kmeans_conditions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: kmeans_conditions.png")

## 13. Heatmap correlazione tra i delta (variabili significative FDR)

Correlazioni Spearman tra i delta scores delle sole variabili significative dopo FDR.  
Risponde alla domanda: *quando il Jerk aumenta sotto workload, aumenta proporzionalmente anche il Path Length? E il Dwell Time?*  
Le stelle indicano la significatività della correlazione (non corretta per confronti multipli — analisi esplorativa).

In [ ]:
from scipy.stats import spearmanr

sig_var_list = sorted(sig_vars)
delta_sig    = delta[sig_var_list].dropna()
n_sv         = len(sig_var_list)

corr_mat = np.zeros((n_sv, n_sv))
pval_mat = np.ones((n_sv, n_sv))
for i, v1 in enumerate(sig_var_list):
    for j, v2 in enumerate(sig_var_list):
        if i == j:
            corr_mat[i, j] = 1.0
        else:
            r, p = spearmanr(delta_sig[v1], delta_sig[v2])
            corr_mat[i, j] = r
            pval_mat[i, j] = p

corr_df = pd.DataFrame(corr_mat, index=sig_var_list, columns=sig_var_list).round(3)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Spearman rho")
ax.set_xticks(range(n_sv))
ax.set_xticklabels(sig_var_list, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(n_sv))
ax.set_yticklabels(sig_var_list, fontsize=9)
for i in range(n_sv):
    for j in range(n_sv):
        r = corr_mat[i, j]
        if i != j:
            p = pval_mat[i, j]
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        else:
            stars = ""
        label = f"{r:.2f}\n{stars}" if stars else f"{r:.2f}"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=8, color="white" if abs(r) > 0.6 else "black")

ax.set_title("Heatmap correlazione Spearman tra delta (variabili sig. FDR)\n* p<0.05  ** p<0.01  *** p<0.001",
             fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "delta_correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: delta_correlation_heatmap.png")
display(corr_df)

## 14. Variabili composite (domain scores)

Approccio suggerito per risolvere la multicollinearità tra le variabili dello stesso dominio funzionale.  
Ogni variabile viene z-scorata sul dataset pooled (36 obs), poi le z-score vengono mediate all'interno di 4 domini tematici + RULA standalone.

| Composite | Variabili | N |
|---|---|---|
| **movement** | SPARC_Dom, Jerk_Dom, VelInv_Dom, Dwell_Dom, Vel_Mean_Dom, Vel_Peak_Dom, Vel_SD_Dom, PathLen_Dom | 8 |
| **wrist** | Wr_Flex_Mean_Dom, Wr_Flex_SD_Dom, Wr_Dev_Mean_Dom, Wr_Dev_SD_Dom | 4 |
| **posture** | Neck_Sag_Mean, Neck_Sag_SD, Trunk_Sag_Mean, Trunk_Sag_SD | 4 |
| **arm** | UA_Mean_Dom, SD_UA_Dom, LA_Mean_Dom, SD_LA_Dom | 4 |
| **standalone** | RULA_Mean | 1 |

Per ogni composite viene calcolato il delta (workload − baseline), testato con paired t-test o Wilcoxon in base alla normalità, e corretto per FDR (5 test invece di 21).  
Per i composite significativi viene analizzata la **direzione** delle variabili interne.

In [ ]:
# ── Definizione domini ────────────────────────────────────────────────────
domains = {
    "movement": ["SPARC_Dom","Jerk_Dom","VelInv_Dom","Dwell_Dom",
                 "Vel_Mean_Dom","Vel_Peak_Dom","Vel_SD_Dom","PathLen_Dom"],
    "wrist":    ["Wr_Flex_Mean_Dom","Wr_Flex_SD_Dom","Wr_Dev_Mean_Dom","Wr_Dev_SD_Dom"],
    "posture":  ["Neck_Sag_Mean","Neck_Sag_SD","Trunk_Sag_Mean","Trunk_Sag_SD"],
    "arm":      ["UA_Mean_Dom","SD_UA_Dom","LA_Mean_Dom","SD_LA_Dom"],
}
standalone_comp = ["RULA_Mean"]
all_vars_comp   = [v for grp in domains.values() for v in grp] + standalone_comp

# ── Z-score sul dataset pooled (36 obs) ──────────────────────────────────
df_z = df_clean.copy()
for var in all_vars_comp:
    mu = df_clean[var].astype(float).mean()
    sd = df_clean[var].astype(float).std(ddof=1)
    df_z[var] = (df_clean[var].astype(float) - mu) / sd

# ── Composite scores (media z-score dentro il dominio) ───────────────────
for domain, vars_list in domains.items():
    df_z[f"comp_{domain}"] = df_z[vars_list].mean(axis=1)
df_z["comp_RULA"] = df_z["RULA_Mean"]

composite_cols   = [f"comp_{d}" for d in domains] + ["comp_RULA"]
composite_labels = list(domains.keys()) + ["RULA"]

# ── Delta per soggetto (workload − baseline) ─────────────────────────────
df_z_b  = df_z[df_z["Condition"]=="baseline"].set_index("Subject_ID")
df_z_wl = df_z[df_z["Condition"]=="workload"].set_index("Subject_ID")
common_z = df_z_b.index.intersection(df_z_wl.index)

delta_comp = pd.DataFrame(index=common_z)
for col in composite_cols:
    delta_comp[col] = df_z_wl.loc[common_z, col] - df_z_b.loc[common_z, col]

# ── Normalità dei delta ───────────────────────────────────────────────────
print("SHAPIRO-WILK sui delta composite")
print("-" * 45)
norm_comp = {}
for col, label in zip(composite_cols, composite_labels):
    d = delta_comp[col].dropna().values
    W, p = shapiro(d)
    norm_comp[col] = p > 0.05
    print(f"  {label:12s}  W={W:.4f}  p={p:.4f}  {'normale' if p>0.05 else 'NON normale'}")

# ── Test effetto condizione ───────────────────────────────────────────────
print("\nEFFETTO CONDIZIONE — composite scores")
print("-" * 45)
comp_results = []
for col, label in zip(composite_cols, composite_labels):
    d   = delta_comp[col].dropna().values
    b_v = df_z_b.loc[common_z, col].values.astype(float)
    w_v = df_z_wl.loc[common_z, col].values.astype(float)
    if norm_comp[col]:
        stat, p   = ttest_rel(w_v, b_v)
        test_name = "t-test"
        eff       = np.mean(d) / np.std(d, ddof=1)
        eff_name  = "d_z"
    else:
        stat, p   = wilcoxon(w_v, b_v, alternative="two-sided")
        test_name = "Wilcoxon"
        eff       = 1 - (2*stat) / (len(d)*(len(d)+1)/2)
        eff_name  = "r_rb"
    comp_results.append({"composite": label, "test": test_name,
                         "stat": round(stat,4), "p_raw": round(p,4),
                         "effect": round(eff,4), "eff_name": eff_name})

comp_df = pd.DataFrame(comp_results)
reject_c, p_fdr_c, _, _ = multipletests(comp_df["p_raw"], method="fdr_bh", alpha=0.05)
comp_df["p_fdr"]   = p_fdr_c.round(4)
comp_df["sig_fdr"] = reject_c
display(comp_df.set_index("composite"))

# ── Direzione variabili interne nei composite significativi ───────────────
sig_domains = [lbl for lbl, sig in zip(composite_labels, reject_c) if sig and lbl in domains]

for label in sig_domains:
    print(f"\nDirezione variabili interne — '{label}'")
    print("-" * 45)
    dir_rows = []
    for var in domains[label]:
        d_var  = (df_z_wl.loc[common_z, var] - df_z_b.loc[common_z, var]).values.astype(float)
        mean_d = np.mean(d_var)
        _, p_sw = shapiro(d_var)
        if p_sw > 0.05:
            _, p_t = ttest_rel(df_z_wl.loc[common_z, var].values.astype(float),
                                df_z_b.loc[common_z, var].values.astype(float))
        else:
            _, p_t = wilcoxon(df_z_wl.loc[common_z, var].values.astype(float),
                               df_z_b.loc[common_z, var].values.astype(float),
                               alternative="two-sided")
        dir_rows.append({"variabile": var,
                         "mean_delta_z": round(mean_d, 4),
                         "direzione": "↑ aumenta" if mean_d > 0 else "↓ diminuisce",
                         "p": round(p_t, 4)})
    dir_df = pd.DataFrame(dir_rows).sort_values("mean_delta_z", key=abs, ascending=False)
    display(dir_df.set_index("variabile"))

if not sig_domains:
    print("\nNessun composite significativo dopo FDR.")

# ── Figura 1: Effect size composite ──────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(8, 4))
colors_comp = ["#2ECC71" if s else "#BDC3C7" for s in reject_c]
ax1.barh(composite_labels, comp_df["effect"], color=colors_comp, edgecolor="white", alpha=0.85)
ax1.axvline(0, color="black", linewidth=0.8)
for i, row in comp_df.iterrows():
    xoff = 0.02 if row["effect"] >= 0 else -0.02
    ha   = "left"  if row["effect"] >= 0 else "right"
    ax1.text(row["effect"]+xoff, i,
             f"q={row['p_fdr']:.3f}" + (" ✓" if row["sig_fdr"] else ""),
             va="center", ha=ha, fontsize=8)
ax1.set_xlabel("Effect size (d_z o r_rb)")
ax1.set_title("Effetto condizione — composite scores\nverde = sig. FDR", fontweight="bold")
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
fig1.tight_layout()
plt.savefig(os.path.join(figures_dir, "composite_effect.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Figura 2: Paired lines per ogni composite ────────────────────────────
fig2, axes2 = plt.subplots(1, len(composite_cols), figsize=(4*len(composite_cols), 4))
for ax2, col, label in zip(axes2, composite_cols, composite_labels):
    b_vals = df_z_b.loc[common_z, col].values.astype(float)
    w_vals = df_z_wl.loc[common_z, col].values.astype(float)
    for bv, wv in zip(b_vals, w_vals):
        ax2.plot([0, 1], [bv, wv],
                 color="#E74C3C" if wv > bv else "#3498DB", alpha=0.45, linewidth=1)
    ax2.boxplot([b_vals, w_vals], positions=[0, 1], widths=0.3,
                medianprops=dict(color="black", linewidth=2))
    ax2.set_xticks([0, 1]); ax2.set_xticklabels(["baseline", "workload"], fontsize=8)
    row = comp_df[comp_df["composite"] == label].iloc[0]
    ax2.set_title(f"{label}{' *' if row['sig_fdr'] else ''}\nq={row['p_fdr']:.3f}",
                  fontweight="bold", fontsize=9)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
fig2.suptitle("Composite scores — baseline vs workload", fontweight="bold")
fig2.tight_layout()
plt.savefig(os.path.join(figures_dir, "composite_paired.png"), dpi=150, bbox_inches="tight")
plt.show()